In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║ Model 1 nested LOOCV + feature stability                           ║
# ║ - 기존 방식(LOO 밖 Top-50 선택) vs nested LOOCV 비교               ║
# ║ - fold별 핵심어 안정성(Jaccard, selection frequency) 계산          ║
# ╚══════════════════════════════════════════════════════════════════════╝

import warnings
warnings.filterwarnings("ignore")

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import LeaveOneOut
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

# ── 전제: 앞 셀에서 df_raw 또는 y_bin, X_full이 정의되어 있어야 함 ──
# 없으면 df_raw 기준으로 자동 생성
try:
    _ = y_bin
except NameError:
    y_bin = (df_raw["위기"].astype(int) >= 1).astype(int).values

try:
    _ = X_full
except NameError:
    X_full = df_raw.drop(columns=["위기", "희귀 단어 (양)", "희귀 단어 (음)"], errors="ignore").astype(float)

years = np.array(df_raw.index.astype(int))
feature_names = X_full.columns.to_list()
X = X_full.values
y = np.array(y_bin).astype(int)

print(f"Data: n={len(y)}, p={X.shape[1]}, positives={int(y.sum())}, negatives={int((1-y).sum())}")

# ── 헬퍼 함수 ─────────────────────────────────────────
def get_metrics(y_true, proba, threshold=0.5):
    pred = (proba >= threshold).astype(int)
    return {
        "auc": float(roc_auc_score(y_true, proba)),
        "acc": float(accuracy_score(y_true, pred)),
        "f1":  float(f1_score(y_true, pred, zero_division=0))
    }

# ══════════════════════════════════════════════════════
# 1) 기존 방식: 전체 데이터로 Top-50 선택 후 LOO-CV
# ══════════════════════════════════════════════════════
scaler_full = StandardScaler()
X_scaled_full = pd.DataFrame(
    scaler_full.fit_transform(X),
    columns=feature_names,
    index=years
)

corr_full = X_scaled_full.corrwith(pd.Series(y.astype(float), index=years)).fillna(0.0)
selected_full = corr_full.abs().nlargest(50).index.to_list()

X_leaky = X_scaled_full[selected_full].values

loo = LeaveOneOut()
proba_original = np.zeros(len(y), dtype=float)

for tr, te in loo.split(X_leaky):
    lr = LogisticRegression(
        penalty="l2", C=0.05, max_iter=3000,
        class_weight="balanced", solver="lbfgs", random_state=42
    )
    lr.fit(X_leaky[tr], y[tr])
    proba_original[te] = lr.predict_proba(X_leaky[te])[:, 1]

m_original = get_metrics(y, proba_original)
print("Original outside-selection LOOCV:", m_original)

# ══════════════════════════════════════════════════════
# 2) Nested LOOCV: 각 fold 안에서만 Top-50 재선택
# ══════════════════════════════════════════════════════
proba_nested = np.zeros(len(y), dtype=float)

# feature selection stability
selected_counts = pd.Series(0, index=feature_names, dtype=int)
selected_each_fold = {}

for tr, te in loo.split(X):
    X_tr_raw, X_te_raw = X[tr], X[te]
    y_tr = y[tr]

    # (a) scaler도 training fold에서만 fit
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr_raw)
    X_te = scaler.transform(X_te_raw)

    # (b) supervised feature selection도 training fold에서만 수행
    X_tr_df = pd.DataFrame(X_tr, columns=feature_names)
    corr = X_tr_df.corrwith(pd.Series(y_tr.astype(float))).fillna(0.0)
    top50 = corr.abs().nlargest(50).index.to_list()

    test_year = int(years[te][0])
    selected_each_fold[test_year] = top50
    selected_counts[top50] += 1

    # (c) 선택된 feature만 사용
    col_idx = [feature_names.index(c) for c in top50]
    X_tr_sel = X_tr[:, col_idx]
    X_te_sel = X_te[:, col_idx]

    lr = LogisticRegression(
        penalty="l2", C=0.05, max_iter=3000,
        class_weight="balanced", solver="lbfgs", random_state=42
    )
    lr.fit(X_tr_sel, y_tr)
    proba_nested[te] = lr.predict_proba(X_te_sel)[:, 1]

m_nested = get_metrics(y, proba_nested)
print("Nested LOOCV:", m_nested)

# ══════════════════════════════════════════════════════
# 3) feature stability 계산
# ══════════════════════════════════════════════════════
selection_stability = selected_counts.sort_values(ascending=False)
top20_stable = selection_stability.head(20)

selected_sets = [set(v) for v in selected_each_fold.values()]
jaccards = []
for i in range(len(selected_sets)):
    for j in range(i + 1, len(selected_sets)):
        a, b = selected_sets[i], selected_sets[j]
        jaccards.append(len(a & b) / len(a | b))

mean_jaccard = float(np.mean(jaccards)) if jaccards else np.nan

print(f"Mean pairwise Jaccard among Top-50 sets: {mean_jaccard:.4f}")
print("\nTop 10 most frequently selected features:")
print(top20_stable.head(10).to_string())

# fold 전체에서 항상 선택된 feature
always_selected = selection_stability[selection_stability == len(y)].index.to_list()
print(f"\nAlways selected in all {len(y)} folds:")
print(always_selected[:20])

# ══════════════════════════════════════════════════════
# 4) 결과 저장
# ══════════════════════════════════════════════════════
pred_df = pd.DataFrame({
    "year": years,
    "y_true": y,
    "proba_original_outside_selection": proba_original,
    "pred_original": (proba_original >= 0.5).astype(int),
    "proba_nested_loocv": proba_nested,
    "pred_nested": (proba_nested >= 0.5).astype(int),
    "delta_nested_minus_original": proba_nested - proba_original,
})
pred_df.to_csv("kangxi_model1_nested_loocv_predictions.csv", index=False, encoding="utf-8-sig")

summary = {
    "n": int(len(y)),
    "n_positive": int(y.sum()),
    "n_negative": int((1 - y).sum()),
    "original_outside_selection": m_original,
    "nested_loocv": m_nested,
    "auc_drop_nested_minus_original": float(m_nested["auc"] - m_original["auc"]),
    "acc_drop_nested_minus_original": float(m_nested["acc"] - m_original["acc"]),
    "f1_drop_nested_minus_original": float(m_nested["f1"] - m_original["f1"]),
    "mean_pairwise_jaccard_top50": float(mean_jaccard),
    "always_selected_features": always_selected,
    "top20_selection_frequency": {k: int(v) for k, v in top20_stable.to_dict().items()}
}
with open("kangxi_model1_nested_loocv_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

# ══════════════════════════════════════════════════════
# 5) 시각화
# ══════════════════════════════════════════════════════
BG = "#FAF7F0"
PANEL = "#F0EBE0"
GRID = "#C8BFB0"
TM = "#0F172A"
TS = "#475569"

BLUE = "#1D4ED8"
AMBER = "#B45309"
RED = "#B91C1C"
GREEN = "#047857"

fig = plt.figure(figsize=(16, 10), facecolor=BG)
ax1 = plt.subplot(2, 1, 1)
ax2 = plt.subplot(2, 1, 2)

for ax in [ax1, ax2]:
    ax.set_facecolor(PANEL)
    for sp in ax.spines.values():
        sp.set_color(GRID)
    ax.grid(axis="y", color=GRID, ls="--", lw=0.6, alpha=0.8)
    ax.tick_params(colors=TM)

# 상단: 예측확률 비교
ax1.plot(
    years, proba_original,
    color=BLUE, lw=2.2, marker="o", ms=4,
    label=f"Original outside-selection AUC={m_original['auc']:.3f}"
)
ax1.plot(
    years, proba_nested,
    color=AMBER, lw=2.2, marker="o", ms=4,
    label=f"Nested LOOCV AUC={m_nested['auc']:.3f}"
)
ax1.axhline(0.5, color=TS, lw=1.0, ls="--", alpha=0.8)

for yr, yy in zip(years, y):
    if yy == 1:
        ax1.axvspan(yr - 0.5, yr + 0.5, color=RED, alpha=0.08, lw=0)

ax1.set_xlim(years.min() - 1, years.max() + 1)
ax1.set_ylim(-0.02, 1.02)
ax1.set_ylabel("Predicted probability", color=TM, fontsize=11)
ax1.set_title("Model 1: original vs nested LOOCV", color=TM, fontsize=14, fontweight="bold")
ax1.legend(facecolor=BG, edgecolor=GRID, fontsize=10)

# 하단: 선택 안정성
sel_show = top20_stable.sort_values(ascending=True)
ax2.barh(sel_show.index, sel_show.values, color=GREEN, alpha=0.85)
ax2.set_xlabel("Number of folds in which feature was selected into Top-50", color=TM, fontsize=11)
ax2.set_title(
    f"Selection stability across nested folds (mean Jaccard={mean_jaccard:.3f})",
    color=TM, fontsize=13, fontweight="bold"
)

fig.suptitle(
    "Nested LOOCV re-evaluation of Model 1\n"
    f"n={len(y)}, positives={int(y.sum())}, "
    f"original AUC={m_original['auc']:.3f}, nested AUC={m_nested['auc']:.3f}",
    color=TM, fontsize=15, fontweight="bold"
)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig("kangxi_model1_nested_loocv_comparison.png", dpi=180, bbox_inches="tight", facecolor=BG)
plt.show()

print("\n✅ 저장 완료:")
print("- kangxi_model1_nested_loocv_predictions.csv")
print("- kangxi_model1_nested_loocv_summary.json")
print("- kangxi_model1_nested_loocv_comparison.png")
